# KofamScan CTS Demo

End-to-end test of the cdm_kofamscan CTS tool registered 2026-05-07.

- **Image:** `ghcr.io/kbaseincubator/cdm_kofamscan:0.1.0@sha256:d6b20eccf4c6bf1b095e530844a8b04dbae5fca85daf0c9b2bdffb0cf10a9a42`
- **Refdata UUID:** `84b31af0-a5a7-4016-906c-9ad9eef34c6a`
- **Refdata file:** `cts-refdata/kofam/2025-04-30/kofam_refdata.tar.gz`
- **Cluster:** `kbase`
- **Output:** `cts/io/jplfaria/output/kofamscan/test/v1`

KofamScan annotates protein sequences with KEGG KO terms using HMM profiles. Input must be **protein FASTA**, not nucleotide.

## 1. Setup

In [1]:
tscli = get_task_service_client()
mincli = get_minio_client()

IMAGE = "ghcr.io/kbaseincubator/cdm_kofamscan:0.1.0@sha256:d6b20eccf4c6bf1b095e530844a8b04dbae5fca85daf0c9b2bdffb0cf10a9a42"
OUTPUT_DIR = "cts/io/jplfaria/output/kofamscan/test/v1"

print(tscli.whoami())

{'user': 'jplfaria', 'roles': [], 'allowed_paths': [{'path': 'cts/io/', 'perm': 'write'}]}


## 2. Find protein FASTA inputs

KofamScan needs proteins (.faa). There are protein files left over from a checkm2 demo run in `cts/io/gavin/test_output/kb_demo/`. We use those as a quick test input.

In [2]:
candidates = []
for o in mincli.list_objects("cts", prefix="io/gavin/test_output/kb_demo", recursive=True):
    if o.object_name.endswith(".faa"):
        candidates.append(f"cts/{o.object_name}")

print(f"Found {len(candidates)} protein FASTA file(s):")
for f in candidates:
    print(f"  {f}")

input_files = candidates

Found 4 protein FASTA file(s):
  cts/io/gavin/test_output/kb_demo/0/protein_files/GCA_000008085.1_ASM808v1_genomic.faa
  cts/io/gavin/test_output/kb_demo/0/protein_files/GCA_000010565.1_ASM1056v1_genomic.faa
  cts/io/gavin/test_output/kb_demo/1/protein_files/GCA_000145985.1_ASM14598v1_genomic.faa
  cts/io/gavin/test_output/kb_demo/1/protein_files/GCA_000147015.1_ASM14701v1_genomic.faa


## 3. Submit KofamScan job

We do a quick run with one container per input file. KofamScan with all 28K KEGG profiles can be slow on large inputs; a small bacterial proteome should finish in 5-15 min per genome.

In [3]:
if not input_files:
    raise RuntimeError("No protein FASTA inputs found. Adjust the prefix above or upload .faa files first.")

job = tscli.submit_job(
    IMAGE,
    input_files,
    OUTPUT_DIR,
    cluster="kbase",
    declobber=True,
    output_mount_point="/out",
    args=[
        "-p", "/ref_data/profiles",
        "-k", "/ref_data/ko_list",
        "-o", "/out/annotations.tsv",
        "-f", "detail-tsv",
        "--cpu", "4",
        "--tmp-dir", "/out/tmp",
        tscli.insert_files(),
    ],
    num_containers=len(input_files),
    cpus=4,
    memory="16GB",
    runtime="PT2H",
)
print("Job ID:", job.id)

Job ID: c7be270e-70e2-4000-a208-d0c91efeb84e


## 4. Monitor (non-blocking)

In [9]:
import threading, json

thread = threading.Thread(
    target=lambda: print(json.dumps(job.wait_for_completion(), indent=4)),
    daemon=True,
)
thread.start()

{
    "id": "c7be270e-70e2-4000-a208-d0c91efeb84e",
    "state": "complete",
    "transition_times": [
        {
            "state": "created",
            "time": "2026-05-12T16:59:58.449000Z"
        },
        {
            "state": "download_submitted",
            "time": "2026-05-12T16:59:58.645000Z"
        },
        {
            "state": "job_submitting",
            "time": "2026-05-12T17:00:11.693000Z"
        },
        {
            "state": "job_submitted",
            "time": "2026-05-12T17:00:27.450000Z"
        },
        {
            "state": "upload_submitting",
            "time": "2026-05-12T17:05:35.395000Z"
        },
        {
            "state": "upload_submitted",
            "time": "2026-05-12T17:05:35.562000Z"
        },
        {
            "state": "complete",
            "time": "2026-05-12T17:06:41.876000Z"
        }
    ],
    "user": "jplfaria",
    "admin_meta": {}
}


In [10]:
# Re-run this cell to poll status
print(json.dumps(job.get_job_status(), indent=2, default=str))

{
  "id": "c7be270e-70e2-4000-a208-d0c91efeb84e",
  "state": "complete",
  "transition_times": [
    {
      "state": "created",
      "time": "2026-05-12T16:59:58.449000Z"
    },
    {
      "state": "download_submitted",
      "time": "2026-05-12T16:59:58.645000Z"
    },
    {
      "state": "job_submitting",
      "time": "2026-05-12T17:00:11.693000Z"
    },
    {
      "state": "job_submitted",
      "time": "2026-05-12T17:00:27.450000Z"
    },
    {
      "state": "upload_submitting",
      "time": "2026-05-12T17:05:35.395000Z"
    },
    {
      "state": "upload_submitted",
      "time": "2026-05-12T17:05:35.562000Z"
    },
    {
      "state": "complete",
      "time": "2026-05-12T17:06:41.876000Z"
    }
  ],
  "user": "jplfaria",
  "admin_meta": {}
}


## 5. Inspect output files

In [11]:
for o in job.get_job()["outputs"]:
    print(o["crc64nvme"], o["file"])

n2mQ8pUGvpc= cts/io/jplfaria/output/kofamscan/test/v1/0/annotations.tsv
DfL8lCGkOxw= cts/io/jplfaria/output/kofamscan/test/v1/0/tmp/tabular/tabular.txt
gANvIuZ5Jk4= cts/io/jplfaria/output/kofamscan/test/v1/1/annotations.tsv
lq5DaACZ1UU= cts/io/jplfaria/output/kofamscan/test/v1/1/tmp/tabular/tabular.txt
ybhogMK9Lk0= cts/io/jplfaria/output/kofamscan/test/v1/2/annotations.tsv
ym6hpe8K/l0= cts/io/jplfaria/output/kofamscan/test/v1/2/tmp/tabular/tabular.txt
fIVcDnaTmEE= cts/io/jplfaria/output/kofamscan/test/v1/3/annotations.tsv
IsCV+TDJ1zk= cts/io/jplfaria/output/kofamscan/test/v1/3/tmp/tabular/tabular.txt


## 6. Read annotations.tsv as a dataframe

`detail-tsv` columns: gene_name, KO, threshold, score, e_value, KO_definition.
Lines starting with `#` are headers/comments; first non-comment line is column names.

In [12]:
import io
import pandas as pd

# Find first annotations.tsv from job outputs
outs = [o for o in job.get_job()["outputs"] if o["file"].endswith("annotations.tsv")]
print(f"{len(outs)} annotations.tsv file(s)")

frames = []
for o in outs:
    bucket, key = o["file"].split("/", 1)
    obj = mincli.get_object(bucket, key)
    raw = obj.read().decode("utf-8")
    # KofamScan detail-tsv: header on first line starting with '#'
    df = pd.read_csv(io.StringIO(raw), sep="\t", comment="#", header=None,
                     names=["gene_name", "KO", "threshold", "score", "e_value", "KO_definition"])
    df["source_file"] = o["file"]
    frames.append(df)

all_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f"Total annotation rows: {len(all_df)}")
print(f"Unique KOs: {all_df['KO'].nunique() if len(all_df) else 0}")
print(f"Unique genes: {all_df['gene_name'].nunique() if len(all_df) else 0}")
all_df.head(20)

4 annotations.tsv file(s)
Total annotation rows: 147510
Unique KOs: 15821
Unique genes: 5094


,gene_name,KO,threshold,score,e_value,KO_definition,source_file
0,AE017199.1_1,K19091,38.33,106.1,1.200000e-31,CRISPR-associated endoribonuclease Cas6 [EC:3....,cts/io/jplfaria/output/kofamscan/test/v1/0/ann...
1,AE017199.1_1,K13533,460.53,11.7,2.100000e-03,"two-component system, sporulation sensor kinas...",cts/io/jplfaria/output/kofamscan/test/v1/0/ann...
2,AE017199.1_2,K03724,281.63,744.2,1.200000e-224,ATP-dependent helicase Lhr and Lhr-like helica...,cts/io/jplfaria/output/kofamscan/test/v1/0/ann...
3,AE017199.1_2,K06877,271.90,209.9,1.900000e-63,DEAD/DEAH box helicase domain-containing protein,cts/io/jplfaria/output/kofamscan/test/v1/0/ann...
4,AE017199.1_2,K03726,494.67,168.1,8.600000e-51,ATP-dependent DNA helicase [EC:5.6.2.4],cts/io/jplfaria/output/kofamscan/test/v1/0/ann...
5,AE017199.1_2,K03654,612.10,126.9,2.900000e-38,ATP-dependent DNA helicase RecQ [EC:5.6.2.4],cts/io/jplfaria/output/kofamscan/test/v1/0/ann...
6,AE017199.1_2,K18692,444.27,106.0,6.200000e-32,ATP-dependent RNA helicase CshB [EC:5.6.2.7],cts/io/jplfaria/output/kofamscan/test/v1/0/ann...
7,AE017199.1_2,K03725,629.97,103.2,3.700000e-31,putative ATP-dependent RNA helicase [EC:5.6.2.-],cts/io/jplfaria/output/kofamscan/test/v1/0/ann...
8,AE017199.1_2,K17679,452.03,99.5,3.900000e-30,"ATP-dependent RNA helicase MSS116, mitochondri...",cts/io/jplfaria/output/kofamscan/test/v1/0/ann...
9,AE017199.1_2,K15271,680.07,99.1,4.600000e-30,ATP-dependent DNA helicase HFM1/MER3 [EC:5.6.2.4],cts/io/jplfaria/output/kofamscan/test/v1/0/ann...


## 7. End-to-end check

If all of the below are True, the registration is fully working and we can close issue #1.

- Job state is `complete`
- At least one `annotations.tsv` exists in `outputs`
- The TSV has annotation rows and KO assignments are recognizable KEGG KOs (start with `K`)

In [13]:
checks = {
    "job complete": job.get_job_status()["state"] == "complete",
    "has annotations.tsv": len(outs) > 0,
    "non-empty": len(all_df) > 0,
    "valid KO format": all_df["KO"].str.startswith("K").all() if len(all_df) else False,
}
for k, v in checks.items():
    print(f"  [{'x' if v else ' '}] {k}")

if all(checks.values()):
    print("\nAll green. Safe to tick the final box on issue #1 and close.")
else:
    print("\nSomething's off. Don't close the issue yet.")

  [x] job complete
  [x] has annotations.tsv
  [x] non-empty
  [x] valid KO format

All green. Safe to tick the final box on issue #1 and close.
